## CHEBII PAMELA JEPKORIR
## **Consumer Behavior Analytics Using Transaction Data**

In [18]:
import pandas as pd
df = pd.read_csv("online_retail_II.csv")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


### 1. BASIC INSPECTION
- Here we check the shape of the dataset to know the number of rows and colums
- Here we also check the information of the dataset to know the exact columns and the data types of the columns
- We also check whether the dataset has missing values

In [19]:
df.shape
df.info()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      541910 non-null  object 
 1   StockCode    541910 non-null  object 
 2   Description  540456 non-null  object 
 3   Quantity     541910 non-null  int64  
 4   InvoiceDate  541910 non-null  object 
 5   Price        541910 non-null  float64
 6   Customer ID  406830 non-null  float64
 7   Country      541910 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64

- The customer ID and Description has 135,080 and 1454 missing values respectively

## **A. EXPLORATORY DATA ANALYSIS**

### 1. Cleaning the dataset

#### a) Remove Missing Customer IDs
- Here all rows with missing customer IDs are dropped

In [20]:
df = df.dropna(subset=['Customer ID'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 406830 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      406830 non-null  object 
 1   StockCode    406830 non-null  object 
 2   Description  406830 non-null  object 
 3   Quantity     406830 non-null  int64  
 4   InvoiceDate  406830 non-null  object 
 5   Price        406830 non-null  float64
 6   Customer ID  406830 non-null  float64
 7   Country      406830 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 27.9+ MB


### 2. **Identification of Cancelled Transactions**
#### a) Inspect Unique Invoice Patterns

In [21]:
df['Invoice'].astype(str).str[:1].value_counts()

Invoice
5    397925
C      8905
Name: count, dtype: int64

- A subset of invoice numbers begins with the letter 'C', indicating a different transaction type from standard numeric invoices.

#### b)  Compare Quantities for Those Invoices
- Check whether these rows contain negative quantities

In [22]:
df[df['Invoice'].astype(str).str.startswith('C')]['Quantity'].describe()

count     8905.000000
mean       -30.859966
std       1170.154939
min     -80995.000000
25%         -6.000000
50%         -2.000000
75%         -1.000000
max         -1.000000
Name: Quantity, dtype: float64

- The invoices with Cancellations have negative quantities
- Comparing it with normal invoces;

In [23]:
df[~df['Invoice'].astype(str).str.startswith('C')]['Quantity'].describe()

count    397925.000000
mean         13.021793
std         180.419984
min           1.000000
25%           2.000000
50%           6.000000
75%          12.000000
max       80995.000000
Name: Quantity, dtype: float64

- The observation show that normal purchases have positive values
- This confirms that invoices starting with 'C' correspond to product returns or cancellations.

#### c) Show Monetary Impact of 'C' Invoices

In [24]:
df[df['Invoice'].astype(str).str.startswith('C')]['Quantity'].sum()

-274808

- The total is negative, further confirming these are returns.

### **Identification of Cancelled Transactions Summary**

After removing records with missing Customer IDs, invoice identifiers were analyzed to understand transaction structure. It was observed that 397,925 invoices began with numeric values, while 8,905 invoices began with the letter 'C'.

Further investigation revealed that invoices beginning with “C” were associated with negative quantities, indicating product returns or cancelled transactions. Since the objective of this study is to analyze actual purchasing behavior and revenue generation, cancelled transactions were excluded from the dataset.

This preprocessing step ensures that customer segmentation, churn prediction, and lifetime value estimation are based solely on completed purchase behavior.


### 3. Removing Negative Quantities
- This step ensures that all negative quantities, including cancellations, are revoved.

In [25]:
df = df[df['Quantity'] > 0]
df.shape

(397925, 8)

### 4. Creating Total Revenue Column

In [28]:
df['TotalPrice'] = df['Quantity'] * df['Price']
print(df.columns)
print(df.head())

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country', 'TotalPrice'],
      dtype='object')
  Invoice StockCode                          Description  Quantity  \
0  536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1  536365     71053                  WHITE METAL LANTERN         6   
2  536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3  536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4  536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

      InvoiceDate  Price  Customer ID         Country  TotalPrice  
0  12/1/2010 8:26   2.55      17850.0  United Kingdom       15.30  
1  12/1/2010 8:26   3.39      17850.0  United Kingdom       20.34  
2  12/1/2010 8:26   2.75      17850.0  United Kingdom       22.00  
3  12/1/2010 8:26   3.39      17850.0  United Kingdom       20.34  
4  12/1/2010 8:26   3.39      17850.0  United Kingdom       20.34  


To quantify the monetary contribution of each transaction, a new variable (TotalPrice) was created by multiplying the quantity purchased by the unit price. This transformation enables accurate computation of customer-level revenue metrics, which are essential for segmentation, churn modeling, and lifetime value estimation.

## **B. Building Customer-Level Features**

- Here transactions are aggregated from transaction level to customer level

- The steps to be made include;

1. Converting InvoiceDate to datetime

2. Defining a reference date (for Recency calculation)

3. Aggregating by Customer ID

### 1. Ensuring InvoiceDate is Datetime

In [29]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


### 2 Defining Snapshot Date
- Recency requires a reference point.

In [32]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)


### 

### 3. Creating RFM Table

In [33]:
rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'Invoice': 'nunique',                                     # Frequency
    'TotalPrice': 'sum'                                        # Monetary
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']